# Eshmun Knowledge Distillation

Compresses a large Eshmun decoder (teacher) into a smaller student model using
**knowledge distillation** (Hinton et al. 2015) with optional intermediate-layer supervision.

**Pipeline:**
1. Install Eshmun from GitHub
2. Login to Weights & Biases and HuggingFace
3. Load teacher, student, tokenizer and dataset
4. (Optional) configure projection layers for hidden-state distillation
5. Configure `DistillationConfig` and build the trainer
6. Train with W&B logging
7. Save / push the student model

## 1. Install dependencies

In [ ]:
!pip install -q git+https://github.com/abidikhairi/eshmun.git
!pip install -q wandb

## 2. Login to W&B and HuggingFace

In [ ]:
import wandb
from huggingface_hub import login as hf_login

wandb.login()
hf_login()  # paste your HF write-access token when prompted

## 3. Imports

In [ ]:
import logging
import re

import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from eshmun.trainer.distillation.config import DistillationConfig
from eshmun.trainer.distillation.trainer import DistillationTrainer

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 4. W&B run configuration

In [ ]:
WANDB_PROJECT  = "eshmun-distillation"
WANDB_RUN_NAME = "eshmun-distill-local-run-1"
OUTPUT_DIR     = "/tmp/eshmun-distill-output"

# HuggingFace repository to push the distilled student to.
HF_REPO_ID = "khairi/eshmun-distilled-student"

## 5. Teacher and student models

The **teacher** is a larger, pre-trained model whose output distribution the student will learn to match.
It is frozen at trainer construction — no gradients flow through it.

The **student** is a smaller model that will be trained. Both must share the same vocabulary.

In [ ]:
TEACHER_ID = "khairi/eshmun-12L-teacher"   # large, frozen
STUDENT_ID = "khairi/6L-ALT-f32-student-SFT"  # small, trainable

tokenizer = AutoTokenizer.from_pretrained(TEACHER_ID)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

teacher = AutoModelForCausalLM.from_pretrained(
    TEACHER_ID,
    device_map="auto",
    trust_remote_code=True,
)

student = AutoModelForCausalLM.from_pretrained(
    STUDENT_ID,
    device_map="auto",
    trust_remote_code=True,
)

n_teacher = sum(p.numel() for p in teacher.parameters())
n_student = sum(p.numel() for p in student.parameters())
print(f"Teacher: {TEACHER_ID}  ({n_teacher / 1e6:.1f}M parameters)")
print(f"Student: {STUDENT_ID}  ({n_student / 1e6:.1f}M parameters)")
print(f"Compression ratio: {n_teacher / n_student:.1f}x")

## 6. Dataset

The dataset must yield dicts with at least:
- `input_ids` — tokenized sequence (LongTensor)
- `labels` — same as `input_ids` with `-100` at positions to ignore for CE loss
- `attention_mask` — optional but recommended

Here we use a SFT-format dataset of protein sequences and apply causal-LM labelling
(labels = input_ids, prompt tokens masked to -100).

In [ ]:
DATASET_ID   = "khairi/swissprot-sft"
TEXT_COLUMN  = "text"          # column containing the full prompt+completion string
MAX_SEQ_LEN  = 512
PROMPT_END_TOKEN = "<|assistant|>"  # tokens before this are masked in labels

raw = load_dataset(DATASET_ID, split="train")

def tokenize(batch):
    enc = tokenizer(
        batch[TEXT_COLUMN],
        max_length=MAX_SEQ_LEN,
        truncation=True,
        padding="max_length",
        return_tensors=None,  # keep as plain lists for HF datasets
    )
    # labels = input_ids; mask padding positions
    labels = [
        [
            token_id if token_id != tokenizer.pad_token_id else -100
            for token_id in ids
        ]
        for ids in enc["input_ids"]
    ]
    enc["labels"] = labels
    return enc

dataset = raw.map(tokenize, batched=True, batch_size=256, remove_columns=raw.column_names)
dataset.set_format("torch")

# 90/10 train-eval split
split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
eval_dataset  = split["test"]

print("Train:", train_dataset)
print("Eval: ", eval_dataset)

## 7. (Optional) Projection layers for hidden-state distillation

When `distill_hidden_states=True` and the student hidden size differs from the teacher
hidden size, you must supply one `nn.Linear` per student layer (no bias) that maps
`student_hidden_size → teacher_hidden_size`.

Skip this cell and set `projections=None` if:
- you are not using `distill_hidden_states`, **or**
- student and teacher have identical hidden sizes.

In [ ]:
import torch.nn as nn

STUDENT_HIDDEN = student.config.hidden_size
TEACHER_HIDDEN = teacher.config.hidden_size
NUM_STUDENT_LAYERS = student.config.num_hidden_layers

projections: dict[str, nn.Linear] | None = None

if STUDENT_HIDDEN != TEACHER_HIDDEN:
    projections = {
        f"layer_{i}": nn.Linear(STUDENT_HIDDEN, TEACHER_HIDDEN, bias=False)
        for i in range(NUM_STUDENT_LAYERS + 1)  # +1 for embedding layer
    }
    # Move projections to the same device as the student
    device = next(student.parameters()).device
    projections = {k: v.to(device) for k, v in projections.items()}
    print(f"Created {len(projections)} projection layers: {STUDENT_HIDDEN} → {TEACHER_HIDDEN}")
else:
    print("Student and teacher hidden sizes match — no projections needed.")

## 8. Trainer with W&B logging and HF Hub push

`DistillationTrainer` logs structured lines at every `logging_steps`. `_WandbHandler`
intercepts those lines and forwards `key=value` metrics to W&B.

`HubDistillationTrainer` overrides `save_model()` to additionally push the student and
tokenizer to the HuggingFace Hub after every local checkpoint save.

In [ ]:
class _WandbHandler(logging.Handler):
    """Parses structured trainer log lines and forwards metrics to W&B."""

    _KV_RE = re.compile(r"(\w+)=([\d.e+\-]+)")

    def emit(self, record: logging.LogRecord) -> None:
        msg = record.getMessage()
        metrics: dict[str, float] = {}
        for match in self._KV_RE.finditer(msg):
            key, raw_val = match.group(1), match.group(2)
            if key == "epoch":
                continue
            try:
                metrics[key] = float(raw_val)
            except ValueError:
                pass
        if not metrics:
            return
        step = int(metrics.pop("step", wandb.run.step))
        wandb.log(metrics, step=step)


_trainer_logger = logging.getLogger("eshmun.trainer.distillation.trainer")
_trainer_logger.addHandler(_WandbHandler())


class HubDistillationTrainer(DistillationTrainer):
    """DistillationTrainer that pushes the student and tokenizer to the HF Hub after every save."""

    def __init__(self, *args, hub_repo_id: str, hub_tokenizer, **kwargs):
        super().__init__(*args, **kwargs)
        self._hub_repo_id = hub_repo_id
        self._hub_tokenizer = hub_tokenizer
        self._save_count = 0

    def save_model(self, output_dir: str | None = None) -> None:
        super().save_model(output_dir)

        self._save_count += 1
        commit_message = f"checkpoint-{self._save_count} (step {output_dir})"

        print(f"Pushing to Hub: {self._hub_repo_id} …")
        self.student.push_to_hub(
            self._hub_repo_id,
            commit_message=commit_message,
            safe_serialization=True,
        )
        self._hub_tokenizer.push_to_hub(
            self._hub_repo_id,
            commit_message=commit_message,
        )
        print(f"  ✓ pushed to https://huggingface.co/{self._hub_repo_id}")

## 9. Train

**Loss breakdown:**

| Term | Weight | Description |
|---|---|---|
| `L_KD` | `alpha` | KL divergence of temperature-scaled logits (soft targets) |
| `L_CE` | `1 - alpha` | Cross-entropy against ground-truth labels (hard targets) |
| `L_hidden` | `hidden_loss_weight` | Layer-wise MSE on hidden states (optional) |
| `L_attn` | `attention_loss_weight` | Layer-wise MSE on attention maps (optional) |

Set `distill_hidden_states=True` / `distill_attentions=True` to enable intermediate-layer supervision.
When the student has fewer layers than the teacher, supply `teacher_layer_map` to select
which teacher layers to align each student layer to.

In [ ]:
distill_config = DistillationConfig(
    output_dir=OUTPUT_DIR,
    # --- general ---
    num_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=100,
    max_grad_norm=1.0,
    logging_steps=10,
    save_steps=500,
    eval_steps=200,
    bf16=True,
    # --- distillation-specific ---
    temperature=4.0,       # soften teacher distribution
    alpha=0.5,             # balance between KD and CE losses
    distill_hidden_states=False,  # set True to add hidden-state MSE
    distill_attentions=False,     # set True to add attention-map MSE
    # teacher_layer_map=[0, 2, 4, 6, 8, 10],  # uncomment if depths differ
    hidden_loss_weight=1.0,
    attention_loss_weight=1.0,
)

run = wandb.init(
    project=WANDB_PROJECT,
    name=WANDB_RUN_NAME,
    config={
        "teacher": TEACHER_ID,
        "student": STUDENT_ID,
        "dataset": DATASET_ID,
        "num_epochs": distill_config.num_epochs,
        "batch_size": distill_config.per_device_train_batch_size,
        "grad_accum": distill_config.gradient_accumulation_steps,
        "learning_rate": distill_config.learning_rate,
        "temperature": distill_config.temperature,
        "alpha": distill_config.alpha,
        "distill_hidden_states": distill_config.distill_hidden_states,
        "distill_attentions": distill_config.distill_attentions,
        "hub_repo_id": HF_REPO_ID,
    },
)

trainer = HubDistillationTrainer(
    student=student,
    teacher=teacher,
    config=distill_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    projections=projections,
    hub_repo_id=HF_REPO_ID,
    hub_tokenizer=tokenizer,
)

metrics = trainer.train()
print("Training complete:", metrics)
wandb.log({f"final/{k}": v for k, v in metrics.items()})

## 10. Save student model

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Student model saved to {OUTPUT_DIR}")

artifact = wandb.Artifact(name="eshmun-distilled-student", type="model")
artifact.add_dir(OUTPUT_DIR)
run.log_artifact(artifact)

wandb.finish()
print("W&B run finished.")